# 02 - Manual training and validation

Train on fictional examples only. Start with a lexical baseline, inspect validation errors, then explicitly enable instruction inference or LoRA training. Do not select settings using the test split.

Use the Empathy Lab kernel. GPU cells are opt-in to avoid accidental downloads or long training runs.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

parents = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in parents if (p / 'data' / 'fictional_support.jsonl').is_file()), None)
if ROOT is None:
    ROOT = next((p / 'workshops' / 'leadership-evidence-lab' for p in parents
                 if (p / 'workshops' / 'leadership-evidence-lab' / 'data' / 'fictional_support.jsonl').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the multi-lora repository.')

PROFILE = 'qwen-local'  # or qwen-large / nemotron-large on approved larger hardware
RUN_INFERENCE = False
RUN_TRAINING = False
RUN_NAME = 'manual-01'  # choose a fresh name for each experiment
ARTIFACTS = ROOT / 'artifacts' / PROFILE / RUN_NAME

In [ ]:
from empathy_lab.baseline import fit_baseline, predict_baseline
from empathy_lab.data import load_dataset
from empathy_lab.evaluation import evaluate

records = load_dataset(ROOT / 'data' / 'fictional_support.jsonl')
training = [row for row in records if row['split'] == 'train']
validation = [row for row in records if row['split'] == 'validation']
baseline = fit_baseline(training)
baseline_predictions = predict_baseline(baseline, validation)
baseline_report = evaluate(validation, baseline_predictions)
print(json.dumps(asdict(baseline_report), indent=2))

## Choose parameters explicitly

Profiles pin the model revision and expose device, dtype, token limits, batch size, seed, learning rate, steps, and LoRA rank. The local 0.5B Qwen profile is the feasibility tier, not a SOTA claim. Larger Qwen and Nemotron require a different machine and a model-license review.

Start with a short smoke run, then use a new run name for meaningful training. A low loss on these small fixtures does not establish real-world empathy recognition.

In [ ]:
from empathy_lab.config import ModelConfig

config = ModelConfig.from_json(ROOT / 'configs' / f'{PROFILE}.json')
# Example manual override; leave test data untouched:
# from dataclasses import replace
# config = replace(config, max_steps=2, learning_rate=0.0001)
print(config)

In [ ]:
if RUN_INFERENCE:
    from empathy_lab.modeling import predict
    predictions = predict(config, validation)
    print(json.dumps(asdict(evaluate(validation, predictions)), indent=2))
else:
    print('Inference disabled. Review profile/hardware and set RUN_INFERENCE=True to run.')

In [ ]:
if RUN_TRAINING:
    from contextlib import chdir

    from empathy_lab.modeling import train
    with chdir(ROOT):
        train(config, training, validation, ARTIFACTS.relative_to(ROOT) / 'adapter')
    print('Adapter saved. Freeze your choices before notebook 03 test evaluation.')
else:
    print('Training disabled. Set RUN_TRAINING=True only after reviewing the data and profile.')

## Review before moving on

- Check completion-only masking: the input/support context must not contribute to supervised loss.
- Compare unadapted and adapted validation outcomes with identical prompts and budgets.
- Malformed JSON is an invalid response, not an empathy level of zero.
- Record changes and compute cost; do not cherry-pick only the best seed.
- Freeze the model configuration and adapter before opening the test evaluation.
- Keep adapters and raw run outputs local. Clear notebook outputs before publication.